# Tutorial 190 — Build and verify a research assistant

The earlier tutorials introduced model setup, source processing, retrieval,
source cards, project chat, execution traces, project types, and tools.
This final tutorial combines those ideas into **one small research
workflow** and finishes with a reproducible evaluation.

We will use four short Markdown notes from a **fictional particle-tracking
pilot**. The measurements are invented for teaching purposes and must
not be cited as scientific findings. Keeping the corpus small makes it
possible to inspect every original source and every retrieved passage.

**What you will do:**

1. Copy a controlled corpus into an independent KíLín project workspace.
2. Read, chunk, index, and retrieve original source text without a model.
3. Test source coverage, assemble verifiable citations, and check one
   numerical comparison against the cited original passages.
4. Optionally use a local language model for a complete RAG answer, saved
   retrieval indexes, and diagnostic traces.

The deterministic sections require **no model download or inference**.
The real-model section is explicitly opt-in and requires cached, compatible
models. If you want to revisit a component, start with
[Tutorial 130](./130_source_cards.ipynb),
[140](./140_project_chat.ipynb),
[150](./150_logging_trace.ipynb), or
[170](./180_create_custom_tool.ipynb).

**Before you begin:** See the [tutorial guide](./README.md) for setup,
local model requirements, and workspace safety.


## 1. Prepare the local KíLín checkout

Use the Python 3.12 environment from Tutorial 10. As in other notebooks,
`prepare_notebook()` may repair missing dependencies via `uv sync`; this
setup step can require network access. Install dependencies beforehand
to run the subsequent examples offline.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "kilin").is_dir():
    root = root.parent
print(f"KíLín root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


In [ ]:
import kilin

print("KíLín version:", kilin.__version__)


## 2. Inspect the four original sources

The bundled corpus lives in
[`data/190_research_pilot/`](./data/190_research_pilot/).
It contains the study plan, measurements from automated and manual
tracking, and an interpretation with limitations. All four files explicitly
identify themselves as fictional teaching material.

These files are read-only **inputs** to the exercise. KíLín will work with
a separate copy under `sandbox/` rather than modifying the originals.


In [ ]:
corpus = root / "tutorials" / "data" / "190_research_pilot"
expected_files = {
    "study_plan.md",
    "automated_tracking.md",
    "manual_tracking.md",
    "interpretation.md",
}
actual_files = {item.name for item in corpus.iterdir()}
assert actual_files == expected_files
for path in sorted(corpus.glob("*.md")):
    title = path.read_text(encoding="utf-8").splitlines()[0]
    print(f"{path.name}: {title}")


## 3. Create a managed literature project

`prepare_project_workspace()` makes a new, independent mirror every time.
It never overwrites an existing project, so rerunning this cell allocates
another numbered workspace. We choose the built-in `literature` type
because the corpus consists of research notes rather than software.

Set `allow_download = false` in the **project's own profile**. We also
disable automatic `scientific_paper` source cards **for this illustrative
corpus only**: the notes are not actual papers, and source-card generation
would otherwise request a separate model. This does not remove or change
KíLín's source-card functionality. See Tutorial 130 for that feature.


In [ ]:
project = kilin.prepare_project_workspace(
    corpus,
    "literature",
    root / "sandbox",
)
print("Project workspace:", project.root)
print("Managed source copy:", project.source)


In [ ]:
from configparser import ConfigParser
from kilin.source_card_policy import (
    SourceCardPolicy,
    write_source_card_policy,
)

profile_path = project.root / "profile.ini"
config = ConfigParser()
config.read(profile_path, encoding="utf-8")
config["models"]["allow_download"] = "false"
with profile_path.open("w", encoding="utf-8") as stream:
    config.write(stream)

write_source_card_policy(
    project.root, SourceCardPolicy(False, None, False)
)
project = kilin.open_project(project.root)
assert not project.profile.allow_download
print("Automatic model downloads:", project.profile.allow_download)
print("Automatic source cards: disabled for these example notes")


### Verify that the originals were copied without modification

The project has its own `sources/` directory and a saved
`source_snapshot.json` record. Comparing source bytes is stronger than
merely checking that the same filenames exist.


In [ ]:
from hashlib import sha256

for name in sorted(expected_files):
    original = (corpus / name).read_bytes()
    mirror = (project.source / name).read_bytes()
    assert sha256(original).digest() == sha256(mirror).digest()
    print(f"Identical copy: {name}")

assert (project.root / "source_snapshot.json").is_file()


## 4. Extract and chunk the managed source text

Use the same production APIs as the earlier tutorials. Markdown headings
establish natural source units, and each chunk retains its original
filename and line range. The word counter below is a **teaching-only
approximation**; a production embedding index uses its model's tokenizer.


In [ ]:
report = kilin.scan_repository(project.source, extensions=[".md"])
assert report["complete"]
assert set(report["selected_files"]) == expected_files
source_files = kilin.read_repository_files(report)
units = kilin.extract_source_units(source_files)

chunks = kilin.chunk_source_units(
    units,
    count_tokens=lambda text: len(text.split()),
    target_tokens=95,
    max_tokens=150,
    overlap_tokens=10,
)
assert chunks
print(f"Sources: {len(source_files)}")
print(f"Natural units: {len(units)}")
print(f"Searchable chunks: {len(chunks)}")


In [ ]:
for chunk in chunks:
    location = (
        f"{chunk.path.as_posix()}:"
        f"{chunk.start_line}-{chunk.end_line}"
    )
    print(f"{location} — {chunk.section or chunk.kind}")


## 5. Build a model-free lexical index

TF-IDF retrieval is deterministic and uses the original text. Unlike
semantic embeddings, it can run without cached model weights. Later,
the optional `RAGSession` builds the full persisted retrieval bundle
with a genuine embedding model; the lexical index below is **in memory**.


In [ ]:
lexical_index = kilin.build_lexical_index(chunks)
sample_question = (
    "What was the automated tracking throughput in trajectories "
    "per minute?"
)
sample_hits = kilin.lexical_search(
    lexical_index, sample_question, count=3
)
for item in sample_hits:
    print(f"{item.score:.3f}  {item.chunk.path}")


## 6. Run reproducible retrieval checks

A small evaluation set names the **expected source**, not the expected
wording of a generated answer. Search each question independently and
check that the best **positive-score** chunk comes from that source.
This is a retrieval smoke test, not a claim of general accuracy: four
intentionally clear queries do not constitute a benchmark.


In [ ]:
retrieval_checks = [
    (
        "design",
        "12 fluorescence microscopy movies at 10 frames per second",
        "study_plan.md",
    ),
    (
        "automated",
        "automated tracking throughput trajectories per minute",
        "automated_tracking.md",
    ),
    (
        "manual",
        "manual tracking median localization error pixels",
        "manual_tracking.md",
    ),
    (
        "limitations",
        "external validation uncertainty confidence interval limitations",
        "interpretation.md",
    ),
]


In [ ]:
best_evidence = {}
for topic, question, expected in retrieval_checks:
    ranked = kilin.lexical_search(lexical_index, question, count=5)
    top = ranked[0]
    assert top.score > 0, f"No lexical match for {topic}"
    actual = top.chunk.path.name
    assert actual == expected, (topic, actual, expected)
    best_evidence[topic] = top.chunk
    print(f"PASS  {topic:12} {actual} ({top.score:.3f})")

source_coverage = {
    chunk.path.name for chunk in best_evidence.values()
}
assert source_coverage == expected_files
print(f"Source coverage: {len(source_coverage)}/{len(expected_files)}")


## 7. Assemble auditable evidence citations

The following citation numbers are local to this **deterministic evidence
bundle**; they are not the citations a future model may choose. Each
number points to a chunk from the managed copy, with its true filename
and line range. Confirm that every excerpt comes from the source bytes.

Source cards can help select papers in a large collection, but source-card
descriptions are **not** themselves answer evidence (Tutorial 130).


In [ ]:
citations = {}
for number, (topic, _, _) in enumerate(retrieval_checks, start=1):
    chunk = best_evidence[topic]
    original_copy = project.source / chunk.path
    full_text = original_copy.read_text(encoding="utf-8")
    assert chunk.text.strip() in full_text
    assert chunk.start_line >= 1
    assert chunk.end_line >= chunk.start_line
    citations[topic] = f"[{number}]"
    location = (
        f"{chunk.path.as_posix()}:"
        f"{chunk.start_line}-{chunk.end_line}"
    )
    print(f"\n[{number}] {location}\n{chunk.text.strip()}")


### Check a derived numerical claim against the cited sources

Extract annotation throughput from the two source passages, rather than
supplying invented values to an answer generator. This is a deliberately
narrow, auditable computation—not a general research-summary algorithm.


In [ ]:
import re

def throughput(topic: str) -> int:
    """Read the annotated rate from its retrieved source passage."""
    passage = best_evidence[topic].text
    match = re.search(
        r"tracking throughput was\s+(\d+)\s+trajectories per minute",
        passage,
    )
    if match is None:
        raise ValueError(f"No supported throughput for {topic}")
    return int(match.group(1))

automated_rate = throughput("automated")
manual_rate = throughput("manual")
ratio = automated_rate / manual_rate
assert ratio == 3
print(
    f"Automated: {automated_rate} trajectories/min "
    f"{citations['automated']}"
)
print(
    f"Manual: {manual_rate} trajectories/min "
    f"{citations['manual']}"
)
print(f"Annotation throughput ratio: {ratio:g}x")
print(
    "Interpret the ratio together with the recorded limitations "
    f"{citations['limitations']}."
)


## 8. Check a question the corpus cannot answer

KíLín's lexical search returns a ranked list even when all scores are
zero. A zero score means **no indexed lexical overlap**, not definitive
proof that a claim is false. Here we deliberately choose unrelated
terms to check that the evaluation does not treat zero-score matches as
positive evidence. A scientific assistant should say when evidence is
insufficient rather than inventing an answer.


In [ ]:
unrelated_question = "quasars exoplanets magnetosphere"
unrelated_hits = kilin.lexical_search(
    lexical_index, unrelated_question, count=len(chunks)
)
supported = [item for item in unrelated_hits if item.score > 0]
assert not supported
print("No positive lexical evidence for:", unrelated_question)
print("This is an overlap check, not a complete absence proof.")


## 9. Inspect the persistent project state

So far we have persisted the **source snapshot and project profile**,
but not the model-based retrieval index. The offline lexical index is
still in memory. The optional full RAG run below will create or reuse
`indexes/retrieval.json`, separate from the original Markdown files.


In [ ]:
print("Workspace:", project.root)
print("Profile:", project.root / "profile.ini")
print("Source snapshot:", project.root / "source_snapshot.json")
print("Managed originals:", project.source)
print("Model-based indexes:", project.indexes)
retrieval_file = project.indexes / "retrieval.json"
print("Full retrieval index exists:", retrieval_file.is_file())


## 10. Optional: run the complete model-based RAG session

**This section is disabled by default.** Set `RUN_REAL_MODEL = True` only
after caching compatible chat and embedding models, as explained in
Tutorials 20, 60, and 140. No download is permitted by this project's
profile. KíLín may also require its separately configured routing model.
If a model is unavailable, leave this section disabled; all earlier
verification remains valid.

`RAGSession` will use the literature project type and its current
`source_first` retrieval policy. We deliberately disabled automatic
scientific-paper cards only for these fictional notes. The real pipeline
builds, fingerprints, and persists its retrieval indexes, selects original
passages, generates an answer, and (when enabled) writes a trace.

Model-generated prose is not automatically correct: manually check that
each numerical claim and citation is supported by the actual evidence.
**Do not treat the fictional numbers as publishable results.**


In [ ]:
RUN_REAL_MODEL = False
ENABLE_TRACE = True
REAL_QUESTION = (
    "Compare the automated and manual tracking results in the "
    "pilot. What limits the interpretation of these results?"
)
print("Real model run enabled:", RUN_REAL_MODEL)


In [ ]:
real_model_completed = False
real_evidence = ()
trace_file = None
index_status = None

if RUN_REAL_MODEL:
    # RAGSession resolves the models from the saved project profile.
    # These defaults must already be cached: downloads are disabled.
    session = kilin.RAGSession(
        project.root,
        trace=kilin.TraceConfig(enabled=ENABLE_TRACE),
    )
    try:
        index_status = session.index_status
        answer = session.ask(REAL_QUESTION)
        real_evidence = tuple(session.turns[-1].evidence)
        if session.trace.enabled:
            trace_file = session.trace.readable_path
        print("Question:", REAL_QUESTION)
        print("Answer:\n", answer)
        print("Index status:", index_status)
        real_model_completed = True
    finally:
        session.close()


In [ ]:
if real_model_completed:
    assert retrieval_file.is_file()
    print("Saved retrieval index:", retrieval_file)
    for item in real_evidence:
        location = (
            f"{item.path}:{item.start_line}-{item.end_line}"
        )
        print(f"[{item.citation}] {location}")
        print(item.text)
    if trace_file is not None:
        print("Saved readable trace:", trace_file)
        assert trace_file.is_file()
else:
    print("Full RAG inference skipped; model-free checks are complete.")


## 11. Review the workflow before using your own data

This capstone checks four complementary properties:

| Property | What was checked | What was not established |
|---|---|---|
| Source fidelity | Managed files match the original teaching data byte for byte | Source truth or scientific validity |
| Retrieval quality | Four expected-source queries find a positive top-ranked passage | Accuracy on unseen questions |
| Evidence provenance | Every local citation points to original source text and line numbers | Whether a generated claim is entailed by that passage |
| Persistent state | A profile and source snapshot exist; optional RAG persists its indexes and trace | Model output reliability or cross-platform reproducibility |

For real scientific data, start by checking provenance and permissions,
choose the appropriate project type, test representative queries, verify
citations, and evaluate failure cases. Add scientific-paper source cards
where appropriate, remembering that cards assist **discovery**, while
original passages provide the evidence. Inspect a full model run's
trace before relying on its results.


In [ ]:
assert len(source_coverage) == 4
assert all(citations.values())
assert (project.root / "profile.ini").is_file()
assert (project.root / "source_snapshot.json").is_file()
print("Capstone model-free checks: PASS")
print("Managed project remains at:", project.root)


## End of the tutorial series

**Previous:** [Tutorial 180 — Create a custom KíLín tool](./180_create_custom_tool.ipynb)

You have now followed one research question from controlled source files
through an independent project workspace, chunking, retrieval tests,
evidence citations, and (optionally) model-based answering and traces.
The key distinction is between **retrieval**, **source provenance**,
**answer generation**, and **independent verification**.

Return to the [tutorial guide](./README.md) to select other topics, or
adapt the workflow to your own local code or scientific documents.
